In [1]:
import pandas as pd
import pandas as pd
from datetime import datetime
import calendar

In [2]:
humidity = pd.read_csv("combined_weekly_humidity.csv")
print(humidity)
rainfall = pd.read_csv("combined_rainfall_data.csv")
print(rainfall.head(15))
print(rainfall.tail(15))
temperature = pd.read_csv("combined_temperature_data.csv")
print(temperature)
fumigation = pd.read_csv("fumigation.csv")
print(fumigation)
patient = pd.read_csv("patient_medical_data.csv")
print(patient)
mosquito_net = pd.read_csv("treated_mosquito_net_distribution.csv")
print(mosquito_net)

         District  Year  Week   Humidity
0     CHAKE-CHAKE  2018     1  89.871429
1     CHAKE-CHAKE  2018     2  90.200000
2     CHAKE-CHAKE  2018     3  89.942857
3     CHAKE-CHAKE  2018     4  90.642857
4     CHAKE-CHAKE  2018     5  90.000000
...           ...   ...   ...        ...
3427         WETE  2023    48  92.371429
3428         WETE  2023    49  90.557143
3429         WETE  2023    50  89.900000
3430         WETE  2023    51  90.228571
3431         WETE  2023    52  90.385714

[3432 rows x 4 columns]
        District           Station  Year Month      Week  Rainfall
0   MAGHARIBI  B  Zanzibar Airport  2023   Jan  1st week       3.8
1   MAGHARIBI  B  Zanzibar Airport  2023   Feb  1st week       0.0
2   MAGHARIBI  B  Zanzibar Airport  2023   Mar  1st week       0.0
3   MAGHARIBI  B  Zanzibar Airport  2023   Apr  1st week      51.6
4   MAGHARIBI  B  Zanzibar Airport  2023   May  1st week      69.6
5   MAGHARIBI  B  Zanzibar Airport  2023   Jun  1st week      96.9
6   MAGHARIBI 

/tmp/ipykernel_75144/1159909342.py:10: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  patient = pd.read_csv("patient_medical_data.csv")


In [10]:
temperature.nunique()

District             2
Station              2
Year                 6
Month               12
Week                 4
Min Temperature    268
Max Temperature    340
dtype: int64

In [14]:
import pandas as pd
from datetime import datetime
import calendar

# Load temperature data
temperature = pd.read_csv("combined_temperature_data.csv")

# Define true districts
PEMBA_DISTRICTS = ['WETE', 'MICHEWENI', 'CHAKE-CHAKE', 'MKOANI']
UNGUJA_DISTRICTS = ['MAGHARIBI  B', 'KUSINI', 'MJINI', 'KASKAZINI B', 'MAGHARIBI  A', 'KASKAZINI A', 'KATI']

# Map island -> districts
ISLAND_TO_DISTRICTS = {
    "PEMBA": PEMBA_DISTRICTS,
    "ZANZIBAR": UNGUJA_DISTRICTS,
    "UNGUJA": UNGUJA_DISTRICTS  # in case
}

# Lowercase normalization
temperature['District'] = temperature['District'].str.upper().str.strip()
temperature['Week'] = temperature['Week'].str.lower().str.strip()
temperature['Month'] = temperature['Month'].str.lower().str.strip()

# Map week string to number in month
week_map = {
    "1st week": 1,
    "2nd week": 2,
    "3rd week": 3,
    "4th week": 4,
    "5th week": 5
}
temperature['Week_in_month'] = temperature['Week'].map(week_map)

# Month name to number
month_map = {
    'jan': 1, 'feb': 2, 'mar': 3, 'apr': 4,
    'may': 5, 'jun': 6, 'jul': 7, 'aug': 8,
    'sep': 9, 'oct': 10, 'nov': 11, 'dec': 12
}
temperature['Month_num'] = temperature['Month'].map(month_map)

# Estimate real date for week (based on Monday)
def get_date(row):
    year = int(row['Year'])
    month = int(row['Month_num'])
    week_in_month = int(row['Week_in_month'])

    # Get all Mondays in month
    mondays = [day for day in range(1, calendar.monthrange(year, month)[1] + 1)
               if datetime(year, month, day).weekday() == 0]
    
    # Pick the nth Monday or last one if not enough
    if week_in_month <= len(mondays):
        day = mondays[week_in_month - 1]
    else:
        day = mondays[-1]
        
    return datetime(year, month, day)

temperature['Est_Date'] = temperature.apply(get_date, axis=1)
temperature['ISO_Week'] = temperature['Est_Date'].dt.isocalendar().week
temperature['ISO_Week'] = temperature['ISO_Week'].astype(int)
temperature['Year'] = temperature['Est_Date'].dt.isocalendar().year  # fix for Jan week crossing

# Expand Zanzibar/Pemba into real districts
expanded_rows = []
for _, row in temperature.iterrows():
    island = row['District']
    if island in ISLAND_TO_DISTRICTS:
        for district in ISLAND_TO_DISTRICTS[island]:
            new_row = row.copy()
            new_row['District'] = district
            expanded_rows.append(new_row)
    else:
        expanded_rows.append(row)

# Finalize expanded DataFrame
final_temp_df = pd.DataFrame(expanded_rows)

# Drop the original string-based 'Week' column to avoid duplication
if 'Week' in final_temp_df.columns:
    final_temp_df = final_temp_df.drop(columns=['Week'])

# Now rename safely
final_temp_df = final_temp_df.rename(columns={"ISO_Week": "Week"})

# Keep only required columns
final_temp_df = final_temp_df[[
    "District", "Station", "Year", "Month", "Week",
    "Min Temperature", "Max Temperature"
]]


# Keep only valid ISO weeks (1-52)
print(final_temp_df)
final_temp_df['Week'] = final_temp_df['Week'].astype(int)
final_temp_df = final_temp_df[final_temp_df['Week'] <= 52]
final_temp_df = final_temp_df.sort_values(by=["District", "Year", "Week"])

# Save and print sample
final_temp_df.to_csv("weekly_temperature_data.csv", index=False)
print(final_temp_df.head())
print("\nShape:", final_temp_df.shape)


         District   Station  Year Month  Week  Min Temperature  \
0    MAGHARIBI  B  Zanzibar  2023   jan     1        26.885714   
0          KUSINI  Zanzibar  2023   jan     1        26.885714   
0           MJINI  Zanzibar  2023   jan     1        26.885714   
0     KASKAZINI B  Zanzibar  2023   jan     1        26.885714   
0    MAGHARIBI  A  Zanzibar  2023   jan     1        26.885714   
..            ...       ...   ...   ...   ...              ...   
526        MKOANI     Pemba  2018   nov    48        24.671429   
527          WETE     Pemba  2018   dec    52        24.842857   
527     MICHEWENI     Pemba  2018   dec    52        24.842857   
527   CHAKE-CHAKE     Pemba  2018   dec    52        24.842857   
527        MKOANI     Pemba  2018   dec    52        24.842857   

     Max Temperature  
0          32.142857  
0          32.142857  
0          32.142857  
0          32.142857  
0          32.142857  
..               ...  
526        31.485714  
527        30.771429  


In [3]:
# Define mapping from '1st week', '2nd week', ... to week offset in month
week_map = {
    "1st week": 1,
    "2nd week": 2,
    "3rd week": 3,
    "4th week": 4,
    "5th week": 5
}

# Normalize and clean text if needed
rainfall['Week'] = rainfall['Week'].str.lower().str.strip()
rainfall['Week_num_in_month'] = rainfall['Week'].map(week_map)

# Month abbreviation to number
month_name_to_num = {
    'jan': 1, 'feb': 2, 'mar': 3, 'apr': 4, 'may': 5, 'jun': 6,
    'jul': 7, 'aug': 8, 'sep': 9, 'oct': 10, 'nov': 11, 'dec': 12
}
rainfall['Month'] = rainfall['Month'].str.lower().str.strip()
rainfall['Month_num'] = rainfall['Month'].map(month_name_to_num)

# Estimate date for each (year, month, week_in_month) by picking first day of week (e.g. Monday)
def get_week_date(row):
    year = int(row['Year'])
    month = int(row['Month_num'])
    week_in_month = int(row['Week_num_in_month'])
    
    # Get all Mondays in that month
    first_day = datetime(year, month, 1)
    last_day = calendar.monthrange(year, month)[1]
    mondays = [day for day in range(1, last_day+1)
               if datetime(year, month, day).weekday() == 0]  # Monday = 0

    if week_in_month <= len(mondays):
        day = mondays[week_in_month - 1]
    else:
        # Fallback: use last Monday if not enough weeks
        day = mondays[-1]

    return datetime(year, month, day)

# Apply date estimation
rainfall['Est_Date'] = rainfall.apply(get_week_date, axis=1)

# Convert to ISO week number
rainfall['ISO_Year'] = rainfall['Est_Date'].dt.isocalendar().year
rainfall['Week'] = rainfall['Est_Date'].dt.isocalendar().week

# Optional: drop helper columns
rainfall.drop(columns=['Week_num_in_month', 'Month_num', 'Est_Date', 'ISO_Year'], inplace=True)

# Final structure
print(rainfall.shape)

# Save it
rainfall.to_csv("rainfall_with_iso_weeks.csv", index=False)

(2880, 6)


In [3]:
import pandas as pd
import numpy as np

# Define districts
DISTRICTS = [
    'WETE', 'MICHEWENI', 'CHAKE-CHAKE', 'MKOANI',
    'MAGHARIBI  B', 'KUSINI', 'MJINI', 'KASKAZINI B',
    'MAGHARIBI  A', 'KASKAZINI A', 'KATI'
]

# Load mosquito net distribution data
mosquito_net = pd.DataFrame({
    'Year': [2018, 2019, 2020, 2021, 2022, 2023],
    'Net distribution': [178072.0, 222374.0, 415012.0, 878264.0, 0.0, 248000.0]
})

# Load fumigation data
fumigation = pd.DataFrame({
    'Year': [2018, 2019, 2020, 2021, 2022, 2023],
    'IRS coverage (%)': [95.0, 95.2, 96.9, 94.7, 97.8, 94.0]
})

# Constants
WEEKS_PER_YEAR = 52
NET_LIFESPAN_WEEKS = 112
TOTAL_DISTRICTS = len(DISTRICTS)

# Prepare a dictionary to store weekly net effect per district
weekly_net_records = []
weekly_fumigation_records = []

# Create weekly range from 2018 to 2024 (because of 112-week lifespan)
full_weeks = pd.date_range(start='2018-01-01', end='2025-01-01', freq='W-MON')

# Create dataframe to track active nets per district
net_effect_df = pd.DataFrame(index=full_weeks)

# Loop over each year and distribute nets across 112 weeks starting from January
for _, row in mosquito_net.iterrows():
    year = int(row['Year'])
    total_nets = float(row['Net distribution'])
    weekly_nets = total_nets / NET_LIFESPAN_WEEKS
    weekly_nets_per_district = weekly_nets / TOTAL_DISTRICTS

    # Create 112-week window starting from the first week of the year
    start_date = pd.Timestamp(f'{year}-01-01')
    active_weeks = pd.date_range(start=start_date, periods=NET_LIFESPAN_WEEKS, freq='W-MON')

    for week in active_weeks:
        if week in net_effect_df.index:
            net_effect_df.loc[week, f'{year}_net'] = weekly_nets_per_district

# Fill NaNs with 0 and sum across all years to get total active nets
net_effect_df.fillna(0, inplace=True)
net_effect_df['Effective_Nets'] = net_effect_df.sum(axis=1)

# Repeat net values across districts
for district in DISTRICTS:
    for date, row in net_effect_df.iterrows():
        weekly_net_records.append({
            'District': district,
            'Year': date.year,
            'Week': date.isocalendar().week,
            'Date': date,
            'Effective_Net_Usage': row['Effective_Nets']
        })

# Prepare weekly fumigation (IRS coverage) – assume evenly spread across each week of the year
for _, row in fumigation.iterrows():
    year = int(row['Year'])
    irs_coverage = float(row['IRS coverage (%)'])

    # Apply evenly across 52 weeks of the year
    for week in range(1, 53):
        for district in DISTRICTS:
            weekly_fumigation_records.append({
                'District': district,
                'Year': year,
                'Week': week,
                'IRS_Coverage': irs_coverage
            })

# Create DataFrames
net_df = pd.DataFrame(weekly_net_records)
irs_df = pd.DataFrame(weekly_fumigation_records)

# Return heads for confirmation

print(net_df)
print(irs_df)

net_df.to_csv("net_with_weeks.csv", index=False)
irs_df.to_csv("fumigation_with_weeks.csv", index=False)

     District  Year  Week       Date  Effective_Net_Usage
0        WETE  2018     1 2018-01-01           144.538961
1        WETE  2018     2 2018-01-08           144.538961
2        WETE  2018     3 2018-01-15           144.538961
3        WETE  2018     4 2018-01-22           144.538961
4        WETE  2018     5 2018-01-29           144.538961
...       ...   ...   ...        ...                  ...
4021     KATI  2024    49 2024-12-02           201.298701
4022     KATI  2024    50 2024-12-09           201.298701
4023     KATI  2024    51 2024-12-16           201.298701
4024     KATI  2024    52 2024-12-23           201.298701
4025     KATI  2024     1 2024-12-30           201.298701

[4026 rows x 5 columns]
          District  Year  Week  IRS_Coverage
0             WETE  2018     1          95.0
1        MICHEWENI  2018     1          95.0
2      CHAKE-CHAKE  2018     1          95.0
3           MKOANI  2018     1          95.0
4     MAGHARIBI  B  2018     1          95.0
...      

In [3]:
import pandas as pd
import numpy as np

# Load all datasets with proper dtype handling
humidity = pd.read_csv("combined_weekly_humidity.csv")
rainfall = pd.read_csv("rainfall_with_iso_weeks.csv")
temperature = pd.read_csv("weekly_temperature_data.csv")
patient = pd.read_csv("patient_medical_data.csv", low_memory=False)  # Handle mixed types
mosquito_net = pd.read_csv("net_with_weeks.csv")
fumigation = pd.read_csv("fumigation_with_weeks.csv")

# 1. Process and combine meteorological data
def process_meteorological_data(humidity, rainfall, temperature):
    # Standardize district names by stripping whitespace and making uppercase
    humidity['District'] = humidity['District'].str.strip().str.upper()
    rainfall['District'] = rainfall['District'].str.strip().str.upper()
    temperature['District'] = temperature['District'].str.strip().str.upper()
    
    # Convert month names to numbers for consistency
    month_map = {'jan':1, 'feb':2, 'mar':3, 'apr':4, 'may':5, 'jun':6,
                 'jul':7, 'aug':8, 'sep':9, 'oct':10, 'nov':11, 'dec':12}
    rainfall['Month'] = rainfall['Month'].str.lower().map(month_map)
    temperature['Month'] = temperature['Month'].str.lower().map(month_map)
    
    # Select relevant columns
    humidity = humidity[['District', 'Year', 'Week', 'Humidity']]
    rainfall = rainfall[['District', 'Year', 'Week', 'Rainfall']]
    temperature = temperature[['District', 'Year', 'Week', 'Min Temperature', 'Max Temperature']]
    
    # Merge meteorological data
    meteo_data = pd.merge(humidity, rainfall, on=['District', 'Year', 'Week'], how='outer')
    meteo_data = pd.merge(meteo_data, temperature, on=['District', 'Year', 'Week'], how='outer')
    
    return meteo_data

meteorological_data = process_meteorological_data(humidity, rainfall, temperature)

# 2. Process and combine malaria control measures
def process_control_measures(mosquito_net, fumigation):
    # Standardize district names
    mosquito_net['District'] = mosquito_net['District'].str.strip().str.upper()
    fumigation['District'] = fumigation['District'].str.strip().str.upper()
    
    # Aggregate net usage by district, year, week
    net_measures = mosquito_net.groupby(['District', 'Year', 'Week'])['Effective_Net_Usage'].mean().reset_index()
    
    # Aggregate fumigation coverage by district, year, week
    fumigation_measures = fumigation.groupby(['District', 'Year', 'Week'])['IRS_Coverage'].mean().reset_index()
    
    # Merge control measures
    control_data = pd.merge(net_measures, fumigation_measures, on=['District', 'Year', 'Week'], how='outer')
    
    return control_data

control_measures = process_control_measures(mosquito_net, fumigation)

# 3. Process malaria surveillance data
def process_surveillance_data(patient):

    # Standardize district names
    patient['District'] = patient['District'].str.strip().str.upper()
    
    # Convert date to datetime and extract year and week if needed
    patient['Date Of Malaria Results'] = pd.to_datetime(patient['Date Of Malaria Results'])
    
    # Calculate week from date if not properly populated
    if patient['Week'].isnull().any():
        patient['Year'] = patient['Date Of Malaria Results'].dt.isocalendar().year
        patient['Week'] = patient['Date Of Malaria Results'].dt.isocalendar().week
    
    # Aggregate positive cases by district, year, week
    positive_cases = patient[patient['Malaria Positive'] == True]
    surveillance_data = positive_cases.groupby(
        ['District', 'Year', 'Week']).size().reset_index(name='Malaria_Cases')
    
    # Create complete grid of all districts, years, and weeks
    all_districts = patient['District'].unique()
    all_years = range(patient['Year'].min(), patient['Year'].max() + 1)
    all_weeks = range(1, 53)
    
    # Create complete index
    full_index = pd.MultiIndex.from_product(
        [all_districts, all_years, all_weeks],
        names=['District', 'Year', 'Week']
    )
    
    # Reindex with complete grid and fill missing with 0
    surveillance_data = surveillance_data.set_index(['District', 'Year', 'Week'])
    surveillance_data = surveillance_data.reindex(full_index, fill_value=0).reset_index()
    
    return surveillance_data

surveillance_data = process_surveillance_data(patient)

# 4. Combine all datasets into final dataset
def create_final_dataset(meteorological_data, control_measures, surveillance_data):
    # First merge surveillance with meteorological data
    final_data = pd.merge(surveillance_data, meteorological_data, 
                         on=['District', 'Year', 'Week'], how='left')
    
    # Then merge with control measures
    final_data = pd.merge(final_data, control_measures, 
                         on=['District', 'Year', 'Week'], how='left')
    
    # Sort by district, year, week for proper filling
    final_data = final_data.sort_values(['District', 'Year', 'Week'])
    
    # Fill missing values appropriately
    # For control measures, assume missing means no intervention
    final_data['Effective_Net_Usage'] = final_data['Effective_Net_Usage'].fillna(0)
    final_data['IRS_Coverage'] = final_data['IRS_Coverage'].fillna(0)
    
    # For meteorological data, use forward/backward fill within districts
    meteorological_cols = ['Humidity', 'Rainfall', 'Min Temperature', 'Max Temperature']
    
    for col in meteorological_cols:
        final_data[col] = final_data.groupby('District')[col].transform(
            lambda x: x.ffill().bfill())
    
    # Create a date column from year and week
    final_data['Date'] = final_data.apply(
        lambda row: pd.to_datetime(f"{int(row['Year'])}-W{int(row['Week'])}-1", format='%Y-W%W-%w'), 
        axis=1)

    
    # Calculate derived features
    final_data['Temperature_Range'] = final_data['Max Temperature'] - final_data['Min Temperature']
    
    return final_data

# try:
# surveillance_data = pd.merge(surveillance_data, surveillance_data, on=['District', 'Year', 'Week', 'Malaria_Cases'], how='left')
final_dataset = create_final_dataset(meteorological_data, control_measures, surveillance_data)

# Save intermediate and final datasets
meteorological_data.to_csv('Meteorological_data.csv', index=False)
control_measures.to_csv('Malaria_control_measures.csv', index=False)
surveillance_data.to_csv('Malaria_surveillance_data.csv', index=False)
final_dataset.to_csv('Final_malaria_prediction_dataset.csv', index=False)

print("Processing completed successfully!")
print("\nMeteorological Data:")
print(meteorological_data.head())
print("\nMalaria Control Measures:")
print(control_measures.head())
print("\nMalaria Surveillance Data:")
print(surveillance_data.head())
print("\nFinal Combined Dataset:")
print(final_dataset.head())

# except Exception as e:
#     print(f"Error occurred: {str(e)}")
#     print("Trying alternative approach...")
    
#     # Alternative approach for the problematic part
#     final_data = pd.merge(surveillance_data, meteorological_data, 
#                          on=['District', 'Year', 'Week'], how='left')
#     final_data = pd.merge(final_data, control_measures, 
#                          on=['District', 'Year', 'Week'], how='left')
    
#     # Fill missing values without groupby if that was causing issues
#     final_data['Effective_Net_Usage'] = final_data['Effective_Net_Usage'].fillna(0)
#     final_data['IRS_Coverage'] = final_data['IRS_Coverage'].fillna(0)
    
#     for col in ['Humidity', 'Rainfall', 'Min Temperature', 'Max Temperature']:
#         final_data[col] = final_data[col].ffill().bfill()
    
#     final_data.to_csv('Final_malaria_prediction_dataset_alternative.csv', index=False)
#     print("Alternative processing completed. Saved dataset.")

               Sn Malaria Case ID Household Island     District  \
0           71368          117360           UNGUJA        MJINI   
1           70973          116949           UNGUJA        MJINI   
2           70974          116949           UNGUJA        MJINI   
3           71369          117360           UNGUJA        MJINI   
4           76392          118937           UNGUJA        MJINI   
...           ...             ...              ...          ...   
152554  NotInvest             NaN            PEMBA  CHAKE-CHAKE   
152555     180949        002696Â¹              NaN          NaN   
152556  NotInvest           10970              NaN        MJINI   
152557     213454           16252           UNGUJA  MAGHARIBI A   
152558  NotInvest           19321              NaN  MAGHARIBI B   

       Household Shehia Date Of Malaria Results  Week  Month  Year  \
0                 AMANI              2018-01-16     3      1  2018   
1                 AMANI              2018-01-08     2  

In [5]:


import pandas as pd
import numpy as np

# Load all datasets with proper dtype handling
humidity = pd.read_csv("combined_weekly_humidity.csv")
rainfall = pd.read_csv("rainfall_with_iso_weeks.csv")
temperature = pd.read_csv("weekly_temperature_data.csv")
patient = pd.read_csv("patient_medical_data.csv", low_memory=False)  # Handle mixed types
mosquito_net = pd.read_csv("net_with_weeks.csv")
fumigation = pd.read_csv("fumigation_with_weeks.csv")

# 1. Process and combine meteorological data
def process_meteorological_data(humidity, rainfall, temperature):
    # Standardize district names by stripping whitespace and making uppercase
    humidity['District'] = humidity['District'].str.strip().str.upper()
    rainfall['District'] = rainfall['District'].str.strip().str.upper()
    temperature['District'] = temperature['District'].str.strip().str.upper()
    
    # Convert month names to numbers for consistency
    month_map = {'jan':1, 'feb':2, 'mar':3, 'apr':4, 'may':5, 'jun':6,
                 'jul':7, 'aug':8, 'sep':9, 'oct':10, 'nov':11, 'dec':12}
    rainfall['Month'] = rainfall['Month'].str.lower().map(month_map)
    temperature['Month'] = temperature['Month'].str.lower().map(month_map)
    
    # Select relevant columns
    humidity = humidity[['District', 'Year', 'Week', 'Humidity']]
    rainfall = rainfall[['District', 'Year', 'Week', 'Rainfall']]
    temperature = temperature[['District', 'Year', 'Week', 'Min Temperature', 'Max Temperature']]
    
    # Merge meteorological data
    meteo_data = pd.merge(humidity, rainfall, on=['District', 'Year', 'Week'], how='outer')
    meteo_data = pd.merge(meteo_data, temperature, on=['District', 'Year', 'Week'], how='outer')
    
    return meteo_data

meteorological_data = process_meteorological_data(humidity, rainfall, temperature)

# 2. Process and combine malaria control measures
def process_control_measures(mosquito_net, fumigation):
    # Standardize district names
    mosquito_net['District'] = mosquito_net['District'].str.strip().str.upper()
    fumigation['District'] = fumigation['District'].str.strip().str.upper()
    
    # Aggregate net usage by district, year, week
    net_measures = mosquito_net.groupby(['District', 'Year', 'Week'])['Effective_Net_Usage'].mean().reset_index()
    
    # Aggregate fumigation coverage by district, year, week
    fumigation_measures = fumigation.groupby(['District', 'Year', 'Week'])['IRS_Coverage'].mean().reset_index()
    
    # Merge control measures
    control_data = pd.merge(net_measures, fumigation_measures, on=['District', 'Year', 'Week'], how='outer')
    
    return control_data

control_measures = process_control_measures(mosquito_net, fumigation)

# 3. Process malaria surveillance data
def process_surveillance_data(patient):
    import numpy as np

    # Standardize district names
    patient['District'] = patient['District'].str.strip().str.upper()

    # Convert date to datetime
    patient['Date Of Malaria Results'] = pd.to_datetime(patient['Date Of Malaria Results'], errors='coerce')

    # Fill in Year and Week if missing
    if patient['Week'].isnull().any() or patient['Year'].isnull().any():
        patient['Year'] = patient['Date Of Malaria Results'].dt.isocalendar().year
        patient['Week'] = patient['Date Of Malaria Results'].dt.isocalendar().week
        patient['Month'] = patient['Date Of Malaria Results'].dt.month

    # Helper columns
    patient['Sex'] = patient['Sex'].str.strip().str.capitalize()
    patient['Occupation'] = patient['Occupation'].fillna('Others').str.strip().str.lower()
    patient['Age In Years'] = pd.to_numeric(patient['Age In Years'], errors='coerce')
    
    # Age group flags
    patient['age_<5'] = patient['Age In Years'] < 5
    patient['age_5_14'] = patient['Age In Years'].between(5, 14, inclusive='both')
    patient['age_15_29'] = patient['Age In Years'].between(15, 29, inclusive='both')
    patient['age_30_44'] = patient['Age In Years'].between(30, 44, inclusive='both')
    patient['age_45+'] = patient['Age In Years'] >= 45

    # Categorical flags
    patient['is_male'] = patient['Sex'] == 'Male'
    patient['is_female'] = patient['Sex'] == 'Female'
    patient['is_index'] = patient['Household Member Type'].str.contains('Index', na=False, case=False)
    patient['is_additional_index'] = patient['Household Member Type'].str.contains('Additional Index', na=False, case=False)
    patient['is_other_member'] = patient['Household Member Type'].str.contains('Other', na=False, case=False)
    patient['is_private'] = patient['Facility Type'].str.contains('private', na=False, case=False)
    patient['is_public'] = patient['Facility Type'].str.contains('public', na=False, case=False)

    # Add occupation category flags
    for occ in ['student', 'health worker', 'police officer', 'watchman', 'farmer', 'fisherman',
                'unemployed', 'housewife', 'business owner']:
        col = f'occ_{occ.replace(" ", "_")}'
        patient[col] = patient['Occupation'].str.contains(occ, na=False)

    # Subset only positive cases for relevant proportions
    positive_cases = patient[patient['Malaria Positive'] == True]

    # Start aggregating
    group_cols = ['District', 'Year', 'Week']
    agg_dict = {
        'Malaria Positive': 'sum',
        'is_male': 'mean',
        'is_female': 'mean',
        'age_<5': 'mean',
        'age_5_14': 'mean',
        'age_15_29': 'mean',
        'age_30_44': 'mean',
        'age_45+': 'mean',
        'is_index': 'sum',
        'is_additional_index': 'sum',
        'is_other_member': 'sum',
        'is_private': 'sum',
        'is_public': 'sum',
        'Household Location - Latitude': 'mean',
        'Household Location - Longitude': 'mean',
    }

    # Add occupation columns to aggregation dictionary
    for occ in ['student', 'health_worker', 'police_officer', 'watchman', 'farmer',
                'fisherman', 'unemployed', 'housewife', 'business_owner']:
        agg_dict[f'occ_{occ}'] = 'sum'

    surveillance_data = positive_cases.groupby(group_cols).agg(agg_dict).reset_index()

    # Rename columns for clarity
    surveillance_data.rename(columns={
        'Malaria Positive': 'true_malaria_cases',
        'is_male': 'prop_male',
        'is_female': 'prop_female',
        'age_<5': 'prop_<5_age',
        'age_5_14': 'prop_5_14_age',
        'age_15_29': 'prop_15_29_age',
        'age_30_44': 'prop_30_44_age',
        'age_45+': 'prop_45+_age',
        'Household Location - Latitude': 'average_latitude',
        'Household Location - Longitude': 'average_longitude'
    }, inplace=True)

    # Create full grid of all combinations
    all_districts = patient['District'].dropna().unique()
    all_years = range(patient['Year'].min(), patient['Year'].max() + 1)
    all_weeks = range(1, 53)

    full_index = pd.MultiIndex.from_product(
        [all_districts, all_years, all_weeks],
        names=['District', 'Year', 'Week']
    )

    # Reindex to ensure completeness
    surveillance_data = surveillance_data.set_index(['District', 'Year', 'Week'])
    surveillance_data = surveillance_data.reindex(full_index, fill_value=0).reset_index()

    # Add month info using mode from patient data (approximation per Week-Year-District)
    patient['Week'] = patient['Week'].astype(int)
    month_lookup = patient.groupby(['District', 'Year', 'Week'])['Month'].agg(
        lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan
    ).reset_index()
    surveillance_data = pd.merge(surveillance_data, month_lookup, on=['District', 'Year', 'Week'], how='left')

    return surveillance_data



surveillance_data = process_surveillance_data(patient)

# 4. Combine all datasets into final dataset
def create_final_dataset(meteorological_data, control_measures, surveillance_data):
    # First merge surveillance with meteorological data
    final_data = pd.merge(surveillance_data, meteorological_data, 
                         on=['District', 'Year', 'Week'], how='left')
    
    # Then merge with control measures
    final_data = pd.merge(final_data, control_measures, 
                         on=['District', 'Year', 'Week'], how='left')
    
    # Sort by district, year, week for proper filling
    final_data = final_data.sort_values(['District', 'Year', 'Week'])
    
    # Fill missing values appropriately
    # For control measures, assume missing means no intervention
    final_data['Effective_Net_Usage'] = final_data['Effective_Net_Usage'].fillna(0)
    final_data['IRS_Coverage'] = final_data['IRS_Coverage'].fillna(0)
    
    # For meteorological data, use forward/backward fill within districts
    meteorological_cols = ['Humidity', 'Rainfall', 'Min Temperature', 'Max Temperature']
    
    for col in meteorological_cols:
        final_data[col] = final_data.groupby('District')[col].transform(
            lambda x: x.ffill().bfill())
    
    # Create a date column from year and week
    final_data['Date'] = final_data.apply(
        lambda row: pd.to_datetime(f"{int(row['Year'])}-W{int(row['Week'])}-1", format='%Y-W%W-%w'), 
        axis=1)

    
    # Calculate derived features
    final_data['Temperature_Range'] = final_data['Max Temperature'] - final_data['Min Temperature']
    
    return final_data

# try:
# surveillance_data = pd.merge(surveillance_data, surveillance_data, on=['District', 'Year', 'Week', 'Malaria_Cases'], how='left')
final_dataset = create_final_dataset(meteorological_data, control_measures, surveillance_data)

# Save intermediate and final datasets
meteorological_data.to_csv('Meteorological_data.csv', index=False)
control_measures.to_csv('Malaria_control_measures.csv', index=False)
surveillance_data.to_csv('Malaria_surveillance_data.csv', index=False)
final_dataset.to_csv('Final_malaria_prediction_dataset.csv', index=False)

print("Processing completed successfully!")
print("\nMeteorological Data:")
print(meteorological_data.head())
print("\nMalaria Control Measures:")
print(control_measures.head())
print("\nMalaria Surveillance Data:")
print(surveillance_data.head())
print("\nFinal Combined Dataset:")
print(final_dataset.head())

# except Exception as e:
#     print(f"Error occurred: {str(e)}")
#     print("Trying alternative approach...")
    
#     # Alternative approach for the problematic part
#     final_data = pd.merge(surveillance_data, meteorological_data, 
#                          on=['District', 'Year', 'Week'], how='left')
#     final_data = pd.merge(final_data, control_measures, 
#                          on=['District', 'Year', 'Week'], how='left')
    
#     # Fill missing values without groupby if that was causing issues
#     final_data['Effective_Net_Usage'] = final_data['Effective_Net_Usage'].fillna(0)
#     final_data['IRS_Coverage'] = final_data['IRS_Coverage'].fillna(0)
    
#     for col in ['Humidity', 'Rainfall', 'Min Temperature', 'Max Temperature']:
#         final_data[col] = final_data[col].ffill().bfill()
    
#     final_data.to_csv('Final_malaria_prediction_dataset_alternative.csv', index=False)
#     print("Alternative processing completed. Saved dataset.")

Processing completed successfully!

Meteorological Data:
      District  Year  Week   Humidity  Rainfall  Min Temperature  \
0  CHAKE-CHAKE  2018     1  89.871429      14.1        24.342857   
1  CHAKE-CHAKE  2018     2  90.200000       1.1        23.800000   
2  CHAKE-CHAKE  2018     3  89.942857       0.0        23.357143   
3  CHAKE-CHAKE  2018     4  90.642857       0.0        22.828571   
4  CHAKE-CHAKE  2018     5  90.000000       NaN              NaN   

   Max Temperature  
0        29.816667  
1        29.642857  
2        30.228571  
3        30.442857  
4              NaN  

Malaria Control Measures:
      District  Year  Week  Effective_Net_Usage  IRS_Coverage
0  CHAKE-CHAKE  2018     1           144.538961          95.0
1  CHAKE-CHAKE  2018     2           144.538961          95.0
2  CHAKE-CHAKE  2018     3           144.538961          95.0
3  CHAKE-CHAKE  2018     4           144.538961          95.0
4  CHAKE-CHAKE  2018     5           144.538961          95.0

Malaria 